## User based financial analysis


In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

def load_data():
    # Load user data
    user_df = pd.read_excel("dummy_user_data.xlsx")
    
    # Load financial rules
    financial_rules_df = pd.read_excel("Updated_Financial_Analytics.xlsx")
    
    return user_df, financial_rules_df

def calculate_budget(salary, marital_status, region, num_children, category_rules):
    # Extract relevant budget allocation percentages
    category_row = category_rules[(category_rules['Salary'] == salary) & (category_rules['Region'] == region) & (category_rules['Category'] == marital_status)]
    
    if category_row.empty:
        needs_percentage, wants_percentage, savings_percentage = 50, 30, 20
    else:
        needs_percentage, wants_percentage, savings_percentage = map(lambda x: sum(map(int, x.strip('%').split('-'))) / 2, 
                                                                    category_row.iloc[0][['Needs (%)', 'Wants (%)', 'Savings (%)']])
    
    # Adjust based on number of children
    needs_percentage += num_children * 5
    wants_percentage -= num_children * 5
    
    # Adjust based on region
    if region == "Tier 1":
        needs_percentage += 5
        wants_percentage -= 5
    elif region == "Tier 4":
        needs_percentage -= 5
        wants_percentage += 5
    
    # Calculate financial allocations
    needs_amount = salary * (needs_percentage / 100)
    wants_amount = salary * (wants_percentage / 100)
    savings_amount = salary * (savings_percentage / 100)
    
    return needs_amount, wants_amount, savings_amount

def display_user_details(selected_user, user_df, category_rules):
    user_row = user_df[user_df['User Name'] == selected_user].iloc[0]
    salary = user_row['Salary']
    marital_status = user_row['Marital Status']
    region = user_row['Region']
    num_children = user_row.get('Children', 0)
    
    needs, wants, savings = calculate_budget(salary, marital_status, region, num_children, category_rules)
    
    output_df = pd.DataFrame({
        "User": [selected_user],
        "Salary": [salary],
        "Marital Status": [marital_status],
        "Region": [region],
        "Number of Children": [num_children],
        "Needs": [needs],
        "Wants": [wants],
        "Savings": [savings]
    })
    
    display(output_df)

def main():
    user_df, category_rules = load_data()
    
    user_dropdown = widgets.Dropdown(
        options=user_df['User Name'].unique(),
        description='Select User:',
        style={'description_width': 'initial'}
    )
    
    output = widgets.Output()
    
    def on_user_change(change):
        with output:
            output.clear_output()
            display_user_details(change.new, user_df, category_rules)
    
    user_dropdown.observe(on_user_change, names='value')
    
    display(user_dropdown, output)

main()


Dropdown(description='Select User:', options=('Lori Bailey', 'Kenneth Yu', 'Jose Phillips', 'Janice Davies', '…

Output()

## Salary based financial analysis

In [6]:
# import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

def load_data():
    # Load financial rules
    financial_rules_df = pd.read_excel("Updated_Financial_Analytics.xlsx")
    return financial_rules_df

def calculate_budget(salary, marital_status, region, num_children, category_rules):
    # Extract relevant budget allocation percentages
    category_row = category_rules[(category_rules['Region'] == region) & (category_rules['Category'] == marital_status)]
    
    if category_row.empty:
        needs_percentage, wants_percentage, savings_percentage = 50, 30, 20
    else:
        needs_percentage, wants_percentage, savings_percentage = map(lambda x: sum(map(int, x.strip('%').split('-'))) / 2, 
                                                                    category_row.iloc[0][['Needs (%)', 'Wants (%)', 'Savings (%)']])
    
    # Adjust based on number of children
    if marital_status in ["Married", "Divorced"] and num_children:
        needs_percentage += num_children * 5
        wants_percentage -= num_children * 5
    
    # Adjust based on region
    if region == "Tier 1":
        needs_percentage += 5
        wants_percentage -= 5
    elif region == "Tier 4":
        needs_percentage -= 5
        wants_percentage += 5
    
    # Calculate financial allocations
    needs_amount = salary * (needs_percentage / 100)
    wants_amount = salary * (wants_percentage / 100)
    savings_amount = salary * (savings_percentage / 100)
    
    return needs_amount, wants_amount, savings_amount

def display_result(salary, marital_status, region, num_children, category_rules, output):
    needs, wants, savings = calculate_budget(salary, marital_status, region, num_children, category_rules)
    
    output_df = pd.DataFrame({
        "Salary": [salary],
        "Marital Status": [marital_status],
        "Region": [region],
        "Number of Children": [num_children],
        "Needs": [needs],
        "Wants": [wants],
        "Savings": [savings]
    })
    
    with output:
        output.clear_output()
        display(output_df)

def main():
    category_rules = load_data()

    # Inject CSS for pastel colors
    display(HTML("""
    <style>
        button[data-value="Single"] { background-color: #FFDDC1 !important; } /* Peach */
        button[data-value="Married"] { background-color: #C1E1C1 !important; } /* Pastel Green */
        button[data-value="Divorced"] { background-color: #ADD8E6 !important; } /* Soft Blue */
        
        button[data-value="Tier 1"] { background-color: #FFC8A2 !important; } /* Soft Orange */
        button[data-value="Tier 2"] { background-color: #D4A5A5 !important; } /* Dusty Rose */
        button[data-value="Tier 3"] { background-color: #B5EAD7 !important; } /* Light Teal */
        button[data-value="Tier 4"] { background-color: #A2D2FF !important; } /* Soft Sky Blue */
    </style>
    """))

    salary_input = widgets.FloatText(
        description="Enter Salary:",
        style={'description_width': 'initial'}
    )
    
    marital_status_buttons = widgets.ToggleButtons(
        options=["Single", "Married", "Divorced"],
        description="Marital Status:",
        style={'description_width': 'initial'}
    )
    
    region_buttons = widgets.ToggleButtons(
        options=["Tier 1", "Tier 2", "Tier 3", "Tier 4"],
        description="Region:",
        style={'description_width': 'initial'}
    )
    
    num_children_input = widgets.IntText(
        description="Number of Children:",
        value=0,
        style={'description_width': 'initial'}
    )
    
    submit_button = widgets.Button(description="Calculate", button_style='warning')
    output = widgets.Output()
    
    marital_status_box = widgets.VBox([])
    region_box = widgets.VBox([])
    children_box = widgets.VBox([])
    submit_box = widgets.VBox([])
    
    def on_salary_submit(change):
        marital_status_box.children = [marital_status_buttons]
    
    def on_marital_status_change(change):
        if change.new in ["Married", "Divorced"]:  # Show for both Married and Divorced
            children_box.children = [num_children_input]
        else:
            children_box.children = []  # Hide for Single
        region_box.children = [region_buttons]
    
    def on_region_selected(change):
        submit_box.children = [submit_button]
    
    def on_submit_button_click(b):
        display_result(salary_input.value, marital_status_buttons.value, region_buttons.value, num_children_input.value, category_rules, output)
    
    salary_input.observe(on_salary_submit, names='value')
    marital_status_buttons.observe(on_marital_status_change, names='value')
    region_buttons.observe(on_region_selected, names='value')
    submit_button.on_click(on_submit_button_click)
    
    display(salary_input, marital_status_box, children_box, region_box, submit_box, output)

main()


FloatText(value=0.0, description='Enter Salary:', style=DescriptionStyle(description_width='initial'))

VBox()

VBox()

VBox()

VBox()

Output()

## Financial analysis based on sub division of needs,wants and savings

### Installing necessary libraries

In [3]:
!pip install tabulate


In [4]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
from tabulate import tabulate

def load_data():
    # Load financial rules
    financial_rules_df = pd.read_excel("Updated_Financial_Analytics.xlsx")
    return financial_rules_df

def calculate_budget(salary, marital_status, region, num_children, category_rules):
    category_row = category_rules[(category_rules['Region'] == region) & (category_rules['Category'] == marital_status)]
    
    if category_row.empty:
        needs_percentage, wants_percentage, savings_percentage = 50, 30, 20
    else:
        needs_percentage, wants_percentage, savings_percentage = map(lambda x: sum(map(int, x.strip('%').split('-'))) / 2, 
                                                                    category_row.iloc[0][['Needs (%)', 'Wants (%)', 'Savings (%)']])
    
    if marital_status in ["Married", "Divorced"] and num_children:
        needs_percentage += num_children * 5
        wants_percentage -= num_children * 5
    
    if region == "Tier 1":
        needs_percentage += 5
        wants_percentage -= 5
    elif region == "Tier 4":
        needs_percentage -= 5
        wants_percentage += 5
    
    needs_amount = salary * (needs_percentage / 100)
    wants_amount = salary * (wants_percentage / 100)
    savings_amount = salary * (savings_percentage / 100)
    
    return needs_amount, wants_amount, savings_amount

def display_result(salary, marital_status, region, num_children, category_rules, output):
    needs, wants, savings = calculate_budget(salary, marital_status, region, num_children, category_rules)
    
    budget_df = pd.DataFrame({
        "Category": [marital_status],
        "Region": [region],
        "Salary": [salary],
        "Needs": [needs],
        "Wants": [wants],
        "Savings": [savings]
    })
    
    needs_breakdown = pd.DataFrame({
        "Category": ["Housing", "Food", "Transportation", "Healthcare", "Debt Payments"],
        "Amount": [needs * 0.4, needs * 0.2, needs * 0.15, needs * 0.1, needs * 0.15]
    })
    
    wants_breakdown = pd.DataFrame({
        "Category": ["Shopping", "Dining Out", "Entertainment", "Travel", "Hobbies"],
        "Amount": [wants * 0.25, wants * 0.25, wants * 0.2, wants * 0.2, wants * 0.1]
    })
    
    savings_breakdown = pd.DataFrame({
        "Category": ["Emergency Fund", "Investments", "Retirement Savings", "Large Purchase Goals"],
        "Amount": [savings * 0.3, savings * 0.4, savings * 0.2, savings * 0.1]
    })
    
    with output:
        output.clear_output()
        print("\n**Budget Plan**")
        print(tabulate(budget_df, headers='keys', tablefmt='grid'))
        print("\n**Needs Breakdown**")
        print(tabulate(needs_breakdown, headers='keys', tablefmt='grid'))
        print("\n**Wants Breakdown**")
        print(tabulate(wants_breakdown, headers='keys', tablefmt='grid'))
        print("\n**Savings Breakdown**")
        print(tabulate(savings_breakdown, headers='keys', tablefmt='grid'))

def main():
    category_rules = load_data()
    
    salary_input = widgets.FloatText(description="Enter Salary:")
    marital_status_buttons = widgets.ToggleButtons(options=["Single", "Married", "Divorced"], description="Marital Status:")
    region_buttons = widgets.ToggleButtons(options=["Tier 1", "Tier 2", "Tier 3", "Tier 4"], description="Region:")
    num_children_input = widgets.IntText(description="Number of Children:", value=0)
    submit_button = widgets.Button(description="Calculate", button_style='warning')
    output = widgets.Output()
    
    marital_status_box = widgets.VBox([])
    region_box = widgets.VBox([])
    children_box = widgets.VBox([])
    submit_box = widgets.VBox([])
    
    def on_salary_submit(change):
        marital_status_box.children = [marital_status_buttons]
    
    def on_marital_status_change(change):
        children_box.children = [num_children_input] if change.new in ["Married", "Divorced"] else []
        region_box.children = [region_buttons]
    
    def on_region_selected(change):
        submit_box.children = [submit_button]
    
    def on_submit_button_click(b):
        display_result(salary_input.value, marital_status_buttons.value, region_buttons.value, num_children_input.value, category_rules, output)
    
    salary_input.observe(on_salary_submit, names='value')
    marital_status_buttons.observe(on_marital_status_change, names='value')
    region_buttons.observe(on_region_selected, names='value')
    submit_button.on_click(on_submit_button_click)
    
    display(salary_input, marital_status_box, children_box, region_box, submit_box, output)

main()

FloatText(value=0.0, description='Enter Salary:')

VBox()

VBox()

VBox()

VBox()

Output()

## dividing the sub categories into sub categories

In [5]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
from tabulate import tabulate

def load_data():
    # Load the corrected financial analytics file
    financial_rules_df = pd.read_excel("corrected_financial_analytics.xlsx")
    return financial_rules_df

def calculate_budget(salary, marital_status, region, num_children, category_rules):
    category_row = category_rules[(category_rules['Region'] == region) & (category_rules['Category'] == marital_status)]
    
    if category_row.empty:
        needs_percentage, wants_percentage, savings_percentage = 50, 30, 20
    else:
        needs_percentage, wants_percentage, savings_percentage = map(
            lambda x: sum(map(int, x.strip('%').split('-'))) / 2, 
            category_row.iloc[0][['Needs (%)', 'Wants (%)', 'Savings (%)']]
        )
    
    if marital_status in ["Married", "Divorced"] and num_children:
        needs_percentage += num_children * 5
        wants_percentage -= num_children * 5
    
    if region == "Tier 1":
        needs_percentage += 5
        wants_percentage -= 5
    elif region == "Tier 4":
        needs_percentage -= 5
        wants_percentage += 5
    
    needs_amount = salary * (needs_percentage / 100)
    wants_amount = salary * (wants_percentage / 100)
    savings_amount = salary * (savings_percentage / 100)
    
    return needs_amount, wants_amount, savings_amount

def display_result(salary, marital_status, region, num_children, category_rules, output):
    needs, wants, savings = calculate_budget(salary, marital_status, region, num_children, category_rules)
    
    budget_df = pd.DataFrame({
        "Category": [marital_status],
        "Region": [region],
        "Salary": [salary],
        "Needs": [needs],
        "Wants": [wants],
        "Savings": [savings]
    })
    
    needs_breakdown = pd.DataFrame({
        "Category": ["Housing", "Food", "Transportation", "Healthcare", "Debt Payments"],
        "Amount": [needs * 0.4, needs * 0.2, needs * 0.15, needs * 0.1, needs * 0.15]
    })
    
    wants_breakdown = pd.DataFrame({
        "Category": ["Shopping", "Dining Out", "Entertainment", "Travel", "Hobbies"],
        "Amount": [wants * 0.25, wants * 0.25, wants * 0.2, wants * 0.2, wants * 0.1]
    })
    
    savings_breakdown = pd.DataFrame({
        "Category": ["Emergency Fund", "Retirement Savings (EPF)", "Investments (Mutual Funds, Stocks)", "Large Purchase Goals"],
        "Amount": [savings * 0.3, savings * 0.4, savings * 0.2, savings * 0.1]
    })
    
    with output:
        output.clear_output()
        print("\n**Budget Plan**")
        print(tabulate(budget_df, headers='keys', tablefmt='grid'))
        print("\n**Needs Breakdown**")
        print(tabulate(needs_breakdown, headers='keys', tablefmt='grid'))
        print("\n**Wants Breakdown**")
        print(tabulate(wants_breakdown, headers='keys', tablefmt='grid'))
        print("\n**Savings Breakdown**")
        print(tabulate(savings_breakdown, headers='keys', tablefmt='grid'))

def main():
    category_rules = load_data()
    
    salary_input = widgets.FloatText(description="Enter Salary:")
    marital_status_buttons = widgets.ToggleButtons(options=["Single", "Married", "Divorced"], description="Marital Status:")
    region_buttons = widgets.ToggleButtons(options=["Tier 1", "Tier 2", "Tier 3", "Tier 4"], description="Region:")
    num_children_input = widgets.IntText(description="Number of Children:", value=0)
    submit_button = widgets.Button(description="Calculate", button_style='warning')
    output = widgets.Output()
    
    marital_status_box = widgets.VBox([])
    region_box = widgets.VBox([])
    children_box = widgets.VBox([])
    submit_box = widgets.VBox([])
    
    def on_salary_submit(change):
        marital_status_box.children = [marital_status_buttons]
    
    def on_marital_status_change(change):
        children_box.children = [num_children_input] if change.new in ["Married", "Divorced"] else []
        region_box.children = [region_buttons]
    
    def on_region_selected(change):
        submit_box.children = [submit_button]
    
    def on_submit_button_click(b):
        display_result(salary_input.value, marital_status_buttons.value, region_buttons.value, num_children_input.value, category_rules, output)
    
    salary_input.observe(on_salary_submit, names='value')
    marital_status_buttons.observe(on_marital_status_change, names='value')
    region_buttons.observe(on_region_selected, names='value')
    submit_button.on_click(on_submit_button_click)
    
    display(salary_input, marital_status_box, children_box, region_box, submit_box, output)

main()


FloatText(value=0.0, description='Enter Salary:')

VBox()

VBox()

VBox()

VBox()

Output()